<a href="https://colab.research.google.com/github/pkocmann/messyverse/blob/main/notebooks/03_pdf-extraktion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# Rechnungen automatisch auslesen -- aus PDF

Im Ordner `erwerbung/` liegen Lieferantenrechnungen als PDF. Dein Auftrag: aus jeder Rechnung die Rechnungs-Nummer, den Lieferanten und den Rechnungsbetrag auslesen und sammeln. Achtung: eine Rechnung enthält bewusst eine Position **ohne** Betrag -- ein typischer Sonderfall.

In [ ]:
# SETUP (Black Box -- einmal ausführen). Holt deine Arbeitskopie des Übungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
os.chdir("/content")                 # erst aus ZIEL heraus -- sonst löscht der Re-Lauf das eigene Arbeitsverzeichnis
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/pkocmann/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(d) for r, _, d in os.walk(ZIEL) if ".git" not in r)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

## Werkzeug nachladen

Zum Lesen von PDF-Text braucht Colab eine kleine Bibliothek.

In [ ]:
!pip install -q pypdf

## Schritt 1 -- die Lage ansehen

Sieh dir den Text einer Rechnung an, damit du weißt, woran die KI die Felder erkennt. Beachte: der Betrag steht im deutschen Format (z.B. `56,80 EUR`).

In [ ]:
import os
from pypdf import PdfReader
print(sorted(os.listdir("erwerbung")))
print(PdfReader("erwerbung/RE-2025-0003.pdf").pages[0].extract_text())

## Schritt 2 -- die KI prompten

> *Lies jede PDF in `erwerbung/` mit pypdf. Zieh aus dem Text die Rechnungs-Nummer (Zeile 'RECHNUNG ...'), den Lieferanten und den Rechnungsbetrag (Zeile 'Rechnungsbetrag: ... EUR', als Zahl -- Achtung deutsches Komma). Baue ein dict `ergebnis`: Rechnungs-Nummer -> {'betrag': float, 'lieferant': str}. Behandle den Fall, dass eine Position '(Betrag fehlt)' enthält.*

In [ ]:
# Code deines KI-Assistenten hier einfügen.
# Ziel: dict `ergebnis` -> 'RE-2025-XXXX': {'betrag': float, 'lieferant': str}


## Schritt 3 -- gegen das Universum prüfen (Black Box)

Versteht auch einen Betrag, der noch als Text mit Komma (`56,80`) ankommt -- dann meldet die Zelle das, statt abzustürzen.

In [ ]:
import json
try:
    soll = json.load(open("loesungen/pdf_extraktion.golden.json"))
except FileNotFoundError:
    print("Bitte zuerst die SETUP-Zelle ausführen (oben).")
else:
    def _zahl(x):
        if isinstance(x, (int, float)): return float(x)
        if not isinstance(x, str): return None
        s = "".join(ch for ch in x if ch.isdigit() or ch in ".,")
        if "," in s: s = s.replace(".", "").replace(",", ".")
        try: return float(s)
        except ValueError: return None
    def _s(x):
        return str(x).strip()
    try:
        ergebnis
    except NameError:
        print("Noch kein `ergebnis`.")
    else:
        if not isinstance(ergebnis, dict):
            print(f"Dein `ergebnis` ist gerade ein {type(ergebnis).__name__}, erwartet wird ein dict Rechnungs-Nr -> {{...}}.")
        else:
            fehlen = [r for r in soll if r not in ergebnis]
            extra = [r for r in ergebnis if r not in soll]
            kein_obj = [r for r in soll if r in ergebnis and not isinstance(ergebnis[r], dict)]
            unklar, betrag_ab, lieferant_ab = [], [], []
            for r in soll:
                if r not in ergebnis or not isinstance(ergebnis[r], dict): continue
                b = _zahl(ergebnis[r].get("betrag"))
                if b is None: unklar.append(r)
                elif round(b, 2) != soll[r]["betrag_eur"]: betrag_ab.append(r)
                if _s(ergebnis[r].get("lieferant")) != _s(soll[r]["lieferant"]): lieferant_ab.append(r)
            if not (fehlen or extra or kein_obj or unklar or betrag_ab or lieferant_ab):
                print(f"OK -- alle {len(soll)} Rechnungen stimmen (Betrag und Lieferant).")
            else:
                if fehlen: print("Diese Rechnungen fehlen:", fehlen)
                if extra: print("Diese Einträge gehören nicht dazu (nur Rechnungen aus erwerbung/, keine Magazin-Scans):", extra)
                if kein_obj: print("Diese Einträge sind kein {...}-Objekt:", kein_obj)
                if unklar: print("Hier ist der Betrag noch kein sauberer Zahlwert (z.B. '14,50' statt 14.50):", unklar)
                if betrag_ab: print("Bei diesen Rechnungen weicht der Betrag ab:", betrag_ab)
                if lieferant_ab: print("Bei diesen Rechnungen weicht der Lieferant ab:", lieferant_ab)

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Prüfzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurück und bitte um eine korrigierte Fassung. Die häufigsten Ursachen sind übersehene Sonderfälle -- leere Einträge, fehlende Felder, ungewohnte Schreibweisen.

## Weiterdenken

Die Rechnungen tragen das Datum in **gemischten Formaten** (15.03.2025, 2025-04-08, 3. Mai 2025). Lass deine KI zusätzlich das Datum auslesen und einheitlich als JJJJ-MM-TT normieren -- die gleiche Aufgabe begegnet dir beim Datei-Sortieren wieder.